# RAPM Data Preprocessing


https://evolving-hockey.com/glossary/regularized-adjusted-plus-minus/

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from collections import defaultdict
from tqdm.auto import tqdm

from IPython.core.magic import register_cell_magic

@register_cell_magic
def skip(line, cell):
    return

## Load MoneyPuck Shot Data

In [2]:
shots_2526 = pd.read_csv("data/shots_2526.csv")
shots_2526.head(10)

,shotID,arenaAdjustedShotDistance,arenaAdjustedXCord,arenaAdjustedXCordABS,arenaAdjustedYCord,arenaAdjustedYCordAbs,averageRestDifference,awayEmptyNet,awayPenalty1Length,awayPenalty1TimeLeft,...,xCordAdjusted,xFroze,xGoal,xPlayContinuedInZone,xPlayContinuedOutsideZone,xPlayStopped,xRebound,xShotWasOnGoal,yCord,yCordAdjusted
0,0,38.000000,-58.0,58.0,-22.0,22.0,0.0,0,0,0,...,58,0.308458,0.017510,0.350448,0.253295,0.035563,0.034727,0.791506,-22,22
1,1,59.000000,-33.0,33.0,-19.0,19.0,0.0,0,0,0,...,33,0.304403,0.036720,0.356975,0.235788,0.025301,0.040814,0.738728,-19,19
2,2,45.000000,55.0,55.0,-29.0,29.0,0.0,0,0,0,...,55,0.306225,0.016030,0.358237,0.263372,0.026669,0.029466,0.831215,-29,-29
3,3,63.000000,35.0,35.0,33.0,33.0,0.0,0,0,0,...,35,0.200191,0.009735,0.439703,0.294931,0.020398,0.035042,0.659939,33,33
4,4,68.000000,30.0,30.0,-34.0,34.0,0.0,0,0,0,...,30,0.219360,0.009977,0.422430,0.307621,0.020837,0.019774,0.681355,-34,-34
5,5,42.000000,-58.0,58.0,28.0,28.0,7.4,0,0,0,...,58,0.332180,0.049217,0.300620,0.245800,0.033867,0.038316,0.767188,28,-28
6,6,44.000000,-47.0,47.0,-12.0,12.0,0.0,0,0,0,...,47,0.233335,0.016412,0.376547,0.284262,0.038632,0.050812,0.801125,-12,12
7,7,13.000000,-79.0,79.0,9.0,9.0,-11.0,0,0,0,...,79,0.113928,0.085066,0.393169,0.298180,0.027855,0.081801,0.808193,9,-9
8,8,8.062258,-82.0,82.0,4.0,4.0,-8.8,0,0,0,...,82,0.141561,0.081322,0.390513,0.296467,0.024592,0.065546,0.748913,4,-4
9,9,6.082763,-88.0,88.0,6.0,6.0,-8.8,0,0,0,...,88,0.225088,0.231689,0.255184,0.146837,0.042034,0.099168,0.817503,6,-6


## Examination

### Size

In [3]:
shots_2526.shape

(119271, 124)

### Columns

In [4]:
shots_2526.columns.tolist()

['shotID',
 'arenaAdjustedShotDistance',
 'arenaAdjustedXCord',
 'arenaAdjustedXCordABS',
 'arenaAdjustedYCord',
 'arenaAdjustedYCordAbs',
 'averageRestDifference',
 'awayEmptyNet',
 'awayPenalty1Length',
 'awayPenalty1TimeLeft',
 'awaySkatersOnIce',
 'awayTeamCode',
 'awayTeamGoals',
 'defendingTeamAverageTimeOnIce',
 'defendingTeamAverageTimeOnIceOfDefencemen',
 'defendingTeamAverageTimeOnIceOfDefencemenSinceFaceoff',
 'defendingTeamAverageTimeOnIceOfForwards',
 'defendingTeamAverageTimeOnIceOfForwardsSinceFaceoff',
 'defendingTeamAverageTimeOnIceSinceFaceoff',
 'defendingTeamDefencemenOnIce',
 'defendingTeamForwardsOnIce',
 'defendingTeamMaxTimeOnIce',
 'defendingTeamMaxTimeOnIceOfDefencemen',
 'defendingTeamMaxTimeOnIceOfDefencemenSinceFaceoff',
 'defendingTeamMaxTimeOnIceOfForwards',
 'defendingTeamMaxTimeOnIceOfForwardsSinceFaceoff',
 'defendingTeamMaxTimeOnIceSinceFaceoff',
 'defendingTeamMinTimeOnIce',
 'defendingTeamMinTimeOnIceOfDefencemen',
 'defendingTeamMinTimeOnIceOfDefen

### Build `shots_df`

This keeps only the columns we actually need for the project.

In [5]:
columns = [
    # Game
    "shotID",
    "game_id",
    "season",
    "period",
    "time",
    "isPlayoffGame",

    # Teams / strength state
    "teamCode",
    "homeTeamCode",
    "awayTeamCode",
    "isHomeTeam",
    "homeSkatersOnIce",
    "awaySkatersOnIce",
    "homeEmptyNet",
    "awayEmptyNet",

    # Score state
    "homeTeamGoals",
    "awayTeamGoals",

    # Shooter
    "shooterPlayerId",
    "shooterName",
    "playerPositionThatDidEvent",

    # Goalie
    "goalieIdForShot",
    "goalieNameForShot",

    # Shot outcome
    "shotWasOnGoal",
    "goal",
    "xGoal",

    # Event context
    "event",
    "lastEventCategory",
    "lastEventTeam",
    "timeSinceLastEvent",
    "timeSinceFaceoff",
    "location",

    # Players on ice
    "shootingTeamForwardsOnIce",
    "shootingTeamDefencemenOnIce",
    "defendingTeamForwardsOnIce",
    "defendingTeamDefencemenOnIce",
]

shots_df = (
    shots_2526.loc[shots_2526["isPlayoffGame"] == 0, columns]
    .copy()
    .reset_index(drop=True)
)

shots_df.head(10)

,shotID,game_id,season,period,time,isPlayoffGame,teamCode,homeTeamCode,awayTeamCode,isHomeTeam,...,event,lastEventCategory,lastEventTeam,timeSinceLastEvent,timeSinceFaceoff,location,shootingTeamForwardsOnIce,shootingTeamDefencemenOnIce,defendingTeamForwardsOnIce,defendingTeamDefencemenOnIce
0,0,20001,2025,1,76,0,FLA,FLA,CHI,1,...,SHOT,FAC,HOME,1,1,AWAYZONE,3,2,3,2
1,1,20001,2025,1,95,0,FLA,FLA,CHI,1,...,SHOT,FAC,HOME,3,3,AWAYZONE,3,2,3,2
2,2,20001,2025,1,104,0,CHI,FLA,CHI,0,...,MISS,FAC,HOME,8,8,HOMEZONE,3,2,3,2
3,3,20001,2025,1,122,0,CHI,FLA,CHI,0,...,SHOT,FAC,AWAY,16,16,HOMEZONE,3,2,3,2
4,4,20001,2025,1,172,0,CHI,FLA,CHI,0,...,SHOT,FAC,AWAY,13,13,HOMEZONE,3,2,3,2
5,5,20001,2025,1,214,0,FLA,FLA,CHI,1,...,SHOT,BLOCK,HOME,3,55,AWAYZONE,3,2,3,2
6,6,20001,2025,1,218,0,FLA,FLA,CHI,1,...,MISS,FAC,HOME,1,1,AWAYZONE,3,2,3,2
7,7,20001,2025,1,246,0,FLA,FLA,CHI,1,...,SHOT,BLOCK,AWAY,11,29,AWAYZONE,3,2,3,2
8,8,20001,2025,1,281,0,FLA,FLA,CHI,1,...,MISS,SHOT,HOME,35,64,AWAYZONE,3,2,3,2
9,9,20001,2025,1,282,0,FLA,FLA,CHI,1,...,SHOT,MISS,HOME,1,65,AWAYZONE,3,2,3,2


### Shape Examination

In [6]:
shots_df.shape

(112011, 34)

### Create Game and Shot Identifiers

Create standardized identifiers for games and shot events.

- `nhl_game_id` creates a season-specific game identifier using the NHL game ID format.
- `shot_event_id` creates a unique identifier for each shot event across the dataset.

In [7]:
# NHL-style game ID
shots_df["nhl_game_id"] = (
    shots_df["season"].astype(int) * 1_000_000
    + shots_df["game_id"].astype(int)
)

# Unique shot ID across the entire dataset
shots_df["shot_event_id"] = (
    shots_df["nhl_game_id"].astype(str)
    + "_"
    + shots_df["shotID"].astype(str)
)

shots_df[
    [
        "shotID",
        "game_id",
        "season",
        "nhl_game_id",
        "shot_event_id"
    ]
].head(10)

,shotID,game_id,season,nhl_game_id,shot_event_id
0,0,20001,2025,2025020001,2025020001_0
1,1,20001,2025,2025020001,2025020001_1
2,2,20001,2025,2025020001,2025020001_2
3,3,20001,2025,2025020001,2025020001_3
4,4,20001,2025,2025020001,2025020001_4
5,5,20001,2025,2025020001,2025020001_5
6,6,20001,2025,2025020001,2025020001_6
7,7,20001,2025,2025020001,2025020001_7
8,8,20001,2025,2025020001,2025020001_8
9,9,20001,2025,2025020001,2025020001_9


### Create Period-Relative Time

Convert cumulative game time into elapsed time within the current period.

Regulation periods are 20 minutes (1,200 seconds), while overtime begins after 3,600 seconds of regulation. Shootout events are handled separately because they do not represent continuous game time in the same way as regulation or overtime.

This period-relative time will later be used to align events with player shifts and lineup stints.

In [8]:
# Start time of each period in cumulative game seconds
period_start = {
    1: 0,
    2: 1200,
    3: 2400,
    4: 3600
}

# Calculate elapsed time within the period
shots_df["period_time"] = (
    shots_df["time"]
    - shots_df["period"].map(period_start)
)

# Shootout events do not have meaningful shift-relative time
shots_df.loc[
    shots_df["period"] >= 5,
    "period_time"
] = pd.NA


period_time_check = (
    shots_df
    .groupby("period", dropna=False)["period_time"]
    .agg(["min", "max", "count"])
)

period_time_check

,min,max,count
period,,,
1,3.0,1200.0,35827
2,2.0,1200.0,38494
3,2.0,1199.0,36016
4,5.0,299.0,1674


### Examine Strength States

Examine the combinations of home and away skaters on the ice for each shot event.

This allows us to identify the strength states represented in the regular-season data before determining which observations should be retained for the even-strength RAPM model.

In [9]:
strength_counts = (
    shots_df
    .groupby(
        ["homeSkatersOnIce", "awaySkatersOnIce"],
        dropna=False
    )
    .size()
    .reset_index(name="n_shots")
    .sort_values("n_shots", ascending=False)
    .reset_index(drop=True)
)

strength_counts

,homeSkatersOnIce,awaySkatersOnIce,n_shots
0,5,5,87511
1,5,4,9068
2,4,5,8393
3,6,5,1605
4,5,6,1602
5,3,3,1350
6,4,4,1229
7,5,3,285
8,3,5,263
9,4,3,226


### Examine Empty-Net Situations at Even Strength

Examine empty-net indicators within the 5v5, 4v4, and 3v3 skater states.

Equal numbers of skaters do not necessarily guarantee that both teams have a goaltender on the ice. Identifying empty-net situations allows these observations to be separated from normal even-strength play before constructing the RAPM sample.

In [10]:
even_strength_empty_net_check = (
    shots_df.loc[
        (
            (shots_df["homeSkatersOnIce"] == 5) &
            (shots_df["awaySkatersOnIce"] == 5)
        )
        |
        (
            (shots_df["homeSkatersOnIce"] == 4) &
            (shots_df["awaySkatersOnIce"] == 4)
        )
        |
        (
            (shots_df["homeSkatersOnIce"] == 3) &
            (shots_df["awaySkatersOnIce"] == 3)
        )
    ]
    .groupby(
        [
            "homeSkatersOnIce",
            "awaySkatersOnIce",
            "homeEmptyNet",
            "awayEmptyNet"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="n_shots")
    .sort_values("n_shots", ascending=False)
    .reset_index(drop=True)
)

even_strength_empty_net_check

,homeSkatersOnIce,awaySkatersOnIce,homeEmptyNet,awayEmptyNet,n_shots
0,5,5,0,0,87476
1,3,3,0,0,1350
2,4,4,0,0,1229
3,5,5,0,1,19
4,5,5,1,0,16


In [11]:
shots_df.loc[
    (shots_df["homeSkatersOnIce"] == 5) &
    (shots_df["awaySkatersOnIce"] == 7)
].T

,106738
shotID,106738
game_id,21248
season,2025
period,3
time,3598
isPlayoffGame,0
teamCode,MTL
homeTeamCode,MTL
awayTeamCode,TBL
isHomeTeam,1


### Remove Empty-Net Situations

Exclude shot events where either team has an empty net.

Empty-net situations involve an extra attacker and do not correspond to the standard strength states used in the RAPM models. Removing these observations also prevents unusual skater counts associated with extra-attacker situations from being treated as normal strength states.

In [12]:
shots_df = (
    shots_df.loc[
        (shots_df["homeEmptyNet"] == 0)
        & (shots_df["awayEmptyNet"] == 0)
    ]
    .copy()
    .reset_index(drop=True)
)

shots_df.shape

(108396, 37)

In [13]:
strength_counts = (
    shots_df
    .groupby(
        ["homeSkatersOnIce", "awaySkatersOnIce"],
        dropna=False
    )
    .size()
    .reset_index(name="n_shots")
    .sort_values("n_shots", ascending=False)
    .reset_index(drop=True)
)

strength_counts

,homeSkatersOnIce,awaySkatersOnIce,n_shots
0,5,5,87476
1,5,4,9051
2,4,5,8377
3,3,3,1350
4,4,4,1229
5,5,3,285
6,3,5,263
7,4,3,217
8,3,4,134
9,6,5,11


### Examine Unusual Strength States

Inspect the remaining shot events involving six skaters after empty-net situations have been removed.

These observations do not correspond to the standard strength states used in the RAPM models and should be examined before deciding whether they represent valid game situations, data-recording issues, or observations that should be excluded.

In [14]:
unusual_strength_df = shots_df.loc[
    (shots_df["homeSkatersOnIce"] > 5)
    | (shots_df["awaySkatersOnIce"] > 5)
].copy()

unusual_strength_df[
    [
        "shot_event_id",
        "game_id",
        "period",
        "period_time",
        "teamCode",
        "homeTeamCode",
        "awayTeamCode",
        "homeSkatersOnIce",
        "awaySkatersOnIce",
        "homeEmptyNet",
        "awayEmptyNet",
        "goalieIdForShot",
        "goalieNameForShot",
        "event",
        "goal"
    ]
]

,shot_event_id,game_id,period,period_time,teamCode,homeTeamCode,awayTeamCode,homeSkatersOnIce,awaySkatersOnIce,homeEmptyNet,awayEmptyNet,goalieIdForShot,goalieNameForShot,event,goal
3708,2025020045_3836,20045,3,591.0,CBJ,CBJ,NJD,6,5,0,0,8474593,Jacob Markstrom,SHOT,0
32505,2025020395_33531,20395,2,1057.0,PHI,NJD,PHI,4,6,0,0,8474593,Jacob Markstrom,SHOT,0
60050,2025020727_62015,20727,1,1016.0,MTL,WSH,MTL,6,5,0,0,8480313,Logan Thompson,SHOT,0
70924,2025020860_73252,20860,3,454.0,COL,DET,COL,6,5,0,0,8475660,Cam Talbot,MISS,0
70925,2025020860_73253,20860,3,455.0,COL,DET,COL,6,5,0,0,8475660,Cam Talbot,GOAL,1
72360,2025020877_74726,20877,1,645.0,OTT,PIT,OTT,5,6,0,0,8481668,Arturs Silovs,SHOT,0
82340,2025020997_85080,20997,3,902.0,MTL,LAK,MTL,6,5,0,0,8475311,Darcy Kuemper,MISS,0
82987,2025021005_85740,21005,1,756.0,NJD,NJD,DET,6,4,0,0,8476434,John Gibson,SHOT,0
92179,2025021115_95227,21115,3,118.0,VGK,DAL,VGK,6,5,0,0,8479193,Casey DeSmith,SHOT,0
96634,2025021168_99839,21168,3,796.0,NSH,TBL,NSH,6,5,0,0,8477992,Jonas Johansson,SHOT,0


### Exclude Unsupported Strength States

Following the removal of observations flagged as empty-net situations,
14 shot events still involved six skaters on at least one team.

All 14 events were recorded as shots by the six-skater team against an
opposing goaltender. These may represent delayed-penalty situations or
inconsistencies in the empty-net indicators.

Because six-skater situations are not included in the specified RAPM
strength states, these observations are excluded.

The retained strength states are:

- Even strength: `5v5`, `4v4`, `3v3`
- Power play / shorthanded: `5v4`, `4v5`, `5v3`, `3v5`, `4v3`, `3v4`

In [15]:
valid_strength_states = {
    (5, 5),
    (4, 4),
    (3, 3),
    (5, 4),
    (4, 5),
    (5, 3),
    (3, 5),
    (4, 3),
    (3, 4),
}

strength_pairs = list(
    zip(
        shots_df["homeSkatersOnIce"],
        shots_df["awaySkatersOnIce"]
    )
)

shots_df = (
    shots_df.loc[
        pd.Series(strength_pairs).isin(valid_strength_states).to_numpy()
    ]
    .copy()
    .reset_index(drop=True)
)

shots_df.shape

(108382, 37)

### Create Strength-State Labels

Create a strength-state label using the number of home and away skaters on the ice.

The retained observations represent the strength states used in the RAPM models:

- Even strength: `5v5`, `4v4`, and `3v3`
- Power play / shorthanded: `5v4`, `4v5`, `5v3`, `3v5`, `4v3`, and `3v4`

These labels will later be used when constructing the strength-state factor variables for the stint-level regressions.

In [16]:
shots_df["strength_state"] = (
    shots_df["homeSkatersOnIce"].astype(int).astype(str)
    + "v"
    + shots_df["awaySkatersOnIce"].astype(int).astype(str)
)

shots_df["strength_state"].value_counts()

strength_state
5v5    87476
5v4     9051
4v5     8377
3v3     1350
4v4     1229
5v3      285
3v5      263
4v3      217
3v4      134
Name: count, dtype: int64

### Examine Score Values on Goal Events

Inspect goal events to determine whether the recorded home and away goal totals represent the score before or after the goal occurred.

This distinction is necessary before constructing score-state variables for the RAPM models.

In [17]:
goal_score_check = (
    shots_df.loc[
        shots_df["goal"] == 1,
        [
            "shot_event_id",
            "period",
            "period_time",
            "teamCode",
            "homeTeamCode",
            "awayTeamCode",
            "isHomeTeam",
            "homeTeamGoals",
            "awayTeamGoals",
            "goal"
        ]
    ]
    .head(20)
)

goal_score_check

,shot_event_id,period,period_time,teamCode,homeTeamCode,awayTeamCode,isHomeTeam,homeTeamGoals,awayTeamGoals,goal
15,2025020001_15,1,603.0,CHI,FLA,CHI,0,0,0,1
17,2025020001_17,1,666.0,FLA,FLA,CHI,1,0,1,1
24,2025020001_24,1,848.0,FLA,FLA,CHI,1,1,1,1
33,2025020001_33,2,69.0,CHI,FLA,CHI,0,2,1,1
66,2025020001_66,3,580.0,FLA,FLA,CHI,1,2,2,1
102,2025020002_104,1,1168.0,PIT,NYR,PIT,0,0,0,1
179,2025020003_183,2,48.0,COL,LAK,COL,0,0,0,1
184,2025020003_188,2,420.0,COL,LAK,COL,0,0,1,1
196,2025020003_200,2,882.0,COL,LAK,COL,0,0,2,1
218,2025020003_222,3,643.0,COL,LAK,COL,0,0,3,1


### Create Score-State Variables

Create score-state variables using the score immediately before each shot event.

Two score-state definitions are used in the RAPM models:

- **CF model:** a 7-level score differential ranging from -3 to +3, with larger leads or deficits capped at ±3.
- **GF and xGF models:** a 3-level score state indicating whether the home team is trailing, tied, or leading.

These variables control for changes in team behavior associated with the game score.

In [18]:
# Home-team goal differential before the event
shots_df["home_score_diff"] = (
    shots_df["homeTeamGoals"]
    - shots_df["awayTeamGoals"]
)

# 7-level score state for CF:
# values below -3 are capped at -3
# values above +3 are capped at +3
shots_df["score_state_cf"] = (
    shots_df["home_score_diff"]
    .clip(lower=-3, upper=3)
    .astype(int)
)

# 3-level score state for GF and xGF
shots_df["score_state_gf_xgf"] = np.select(
    [
        shots_df["home_score_diff"] < 0,
        shots_df["home_score_diff"] == 0,
        shots_df["home_score_diff"] > 0
    ],
    [
        "trailing",
        "tied",
        "leading"
    ],
    default=None
)

shots_df.head(10)

,shotID,game_id,season,period,time,isPlayoffGame,teamCode,homeTeamCode,awayTeamCode,isHomeTeam,...,shootingTeamDefencemenOnIce,defendingTeamForwardsOnIce,defendingTeamDefencemenOnIce,nhl_game_id,shot_event_id,period_time,strength_state,home_score_diff,score_state_cf,score_state_gf_xgf
0,0,20001,2025,1,76,0,FLA,FLA,CHI,1,...,2,3,2,2025020001,2025020001_0,76.0,5v5,0,0,tied
1,1,20001,2025,1,95,0,FLA,FLA,CHI,1,...,2,3,2,2025020001,2025020001_1,95.0,5v5,0,0,tied
2,2,20001,2025,1,104,0,CHI,FLA,CHI,0,...,2,3,2,2025020001,2025020001_2,104.0,5v5,0,0,tied
3,3,20001,2025,1,122,0,CHI,FLA,CHI,0,...,2,3,2,2025020001,2025020001_3,122.0,5v5,0,0,tied
4,4,20001,2025,1,172,0,CHI,FLA,CHI,0,...,2,3,2,2025020001,2025020001_4,172.0,5v5,0,0,tied
5,5,20001,2025,1,214,0,FLA,FLA,CHI,1,...,2,3,2,2025020001,2025020001_5,214.0,5v5,0,0,tied
6,6,20001,2025,1,218,0,FLA,FLA,CHI,1,...,2,3,2,2025020001,2025020001_6,218.0,5v5,0,0,tied
7,7,20001,2025,1,246,0,FLA,FLA,CHI,1,...,2,3,2,2025020001,2025020001_7,246.0,5v5,0,0,tied
8,8,20001,2025,1,281,0,FLA,FLA,CHI,1,...,2,3,2,2025020001,2025020001_8,281.0,5v5,0,0,tied
9,9,20001,2025,1,282,0,FLA,FLA,CHI,1,...,2,3,2,2025020001,2025020001_9,282.0,5v5,0,0,tied


In [19]:
print("CF score states:")
print(
    shots_df["score_state_cf"]
    .value_counts()
    .sort_index()
)

print("\nGF/xGF score states:")
print(
    shots_df["score_state_gf_xgf"]
    .value_counts()
)

CF score states:
score_state_cf
-3     5660
-2     8142
-1    18939
 0    39585
 1    19796
 2     9132
 3     7128
Name: count, dtype: int64

GF/xGF score states:
score_state_gf_xgf
tied        39585
leading     36056
trailing    32741
Name: count, dtype: int64


## Identify Skaters and Goaltenders

Before constructing on-ice stints, players in the NHL shift data must be separated into skaters and goaltenders.

Goaltender IDs are identified using the `goalieIdForShot` field in the shot data. Players appearing as a goaltender on at least one shot are classified as goalies in the shift data.

This distinction is necessary because skaters and goaltenders enter the RAPM models differently. Skaters receive offensive and defensive player indicators, while goaltenders are included separately as defensive indicators in the GF model.

In [20]:
# Identify players who appeared as a goalie in the shot data
goalie_ids = set(
    shots_df.loc[
        shots_df["goalieIdForShot"].notna()
        & (shots_df["goalieIdForShot"] != 0),
        "goalieIdForShot"
    ]
    .astype(int)
    .unique()
)

len(goalie_ids)

98

## Shift Data

In [22]:
shifts_df = pd.read_csv("data/shifts_2526_clean.csv")

print(shifts_df.shape)
shifts_df.head()


(991779, 25)


,id,detailCode,duration,endTime,eventDescription,eventDetails,eventNumber,firstName,gameId,hexValue,...,teamAbbrev,teamId,teamName,typeCode,start_time,end_time,period_start_time,period_end_time,recorded_duration,calculated_duration
0,15536563,0.0,00:34,00:34,NaN,NaN,6.0,Brad,2025020001,#041E42,...,FLA,13,Florida Panthers,517,0,34,0,34,34.0,34
1,15536606,0.0,01:15,04:18,NaN,NaN,154.0,Brad,2025020001,#041E42,...,FLA,13,Florida Panthers,517,183,258,183,258,75.0,75
2,15536607,0.0,00:19,06:35,NaN,NaN,207.0,Brad,2025020001,#041E42,...,FLA,13,Florida Panthers,517,376,395,376,395,19.0,19
3,15536608,0.0,01:02,08:00,NaN,NaN,216.0,Brad,2025020001,#041E42,...,FLA,13,Florida Panthers,517,418,480,418,480,62.0,62
4,15536609,0.0,00:26,10:03,NaN,NaN,255.0,Brad,2025020001,#041E42,...,FLA,13,Florida Panthers,517,577,603,577,603,26.0,26


In [23]:
shifts_df["is_goalie"] = (
    shifts_df["playerId"]
    .astype(int)
    .isin(goalie_ids)
)

shifts_df["is_goalie"].value_counts()

is_goalie
False    981295
True      10484
Name: count, dtype: int64

In [26]:
player_type_check = (
    shifts_df[
        ["playerId", "firstName", "lastName", "is_goalie"]
    ]
    .drop_duplicates(subset="playerId")
    ["is_goalie"]
    .value_counts()
)

player_type_check

is_goalie
False    943
True      98
Name: count, dtype: int64

In [27]:
goalies_df = (
    shifts_df.loc[
        shifts_df["is_goalie"],
        [
            "playerId",
            "firstName",
            "lastName",
            "teamAbbrev"
        ]
    ]
    .drop_duplicates()
    .sort_values(["lastName", "firstName"])
    .reset_index(drop=True)
)

print("Unique goalie IDs:", goalies_df["playerId"].nunique())

goalies_df.head(20)

Unique goalie IDs: 98


,playerId,firstName,lastName,teamAbbrev
0,8482515,Artur,Akhtyamov,TOR
1,8474596,Jake,Allen,NJD
2,8475883,Frederik,Andersen,CAR
3,8481020,Justus,Annunen,NSH
4,8482137,Yaroslav,Askarov,SJS
5,8476412,Jordan,Binnington,STL
6,8478406,Mackenzie,Blackwood,COL
7,8475683,Sergei,Bobrovsky,FLA
8,8476316,Laurent,Brossoit,SJS
9,8483548,Brandon,Bussi,CAR


### Separate Skater and Goaltender Shifts

The shift data contain 98 unique goaltenders and 943 unique non-goaltenders.

The shift records are separated into skater and goaltender datasets before constructing on-ice stints. Skater shifts will be used to identify changes in the offensive and defensive skaters on the ice, while goaltender information is retained separately for use in the GF RAPM model.

In [28]:
skater_shifts_df = (
    shifts_df.loc[
        ~shifts_df["is_goalie"]
    ]
    .copy()
    .reset_index(drop=True)
)

goalie_shifts_df = (
    shifts_df.loc[
        shifts_df["is_goalie"]
    ]
    .copy()
    .reset_index(drop=True)
)

print("Skater shift records:", len(skater_shifts_df))
print("Goalie shift records:", len(goalie_shifts_df))

print(
    "Unique skaters:",
    skater_shifts_df["playerId"].nunique()
)

print(
    "Unique goalies:",
    goalie_shifts_df["playerId"].nunique()
)

Skater shift records: 981295
Goalie shift records: 10484
Unique skaters: 943
Unique goalies: 98


## Construct No-Substitution Stints

RAPM observations are constructed as stints: continuous intervals of play during which no skater substitutions occur.

For each game and period, every skater shift start and end time is treated as a potential lineup-change boundary. Consecutive boundaries define candidate stints.

The skaters on the ice during each candidate interval will then be identified from the shift data. This allows each stint to represent a constant on-ice lineup.

In [29]:
stint_boundaries = (
    pd.concat(
        [
            skater_shifts_df[
                ["gameId", "period", "period_start_time"]
            ].rename(
                columns={"period_start_time": "boundary_time"}
            ),

            skater_shifts_df[
                ["gameId", "period", "period_end_time"]
            ].rename(
                columns={"period_end_time": "boundary_time"}
            )
        ],
        ignore_index=True
    )
    .drop_duplicates()
    .sort_values(
        ["gameId", "period", "boundary_time"]
    )
    .reset_index(drop=True)
)

stint_boundaries.head(20)

,gameId,period,boundary_time
0,2025020001,1,0
1,2025020001,1,34
2,2025020001,1,39
3,2025020001,1,57
4,2025020001,1,62
5,2025020001,1,65
6,2025020001,1,66
7,2025020001,1,75
8,2025020001,1,92
9,2025020001,1,106


In [30]:
print(
    "Unique games:",
    stint_boundaries["gameId"].nunique()
)

print(
    "Candidate boundaries:",
    len(stint_boundaries)
)

stint_boundaries.groupby("period")["boundary_time"].agg(
    ["min", "max", "count"]
)

Unique games: 1312
Candidate boundaries: 524552


,min,max,count
period,,,
1,0,1200,181432
2,0,1200,160195
3,0,1200,177554
4,0,300,5371


In [32]:
skater_boundary_keys = set(
    zip(
        stint_boundaries["gameId"],
        stint_boundaries["period"],
        stint_boundaries["boundary_time"]
    )
)

goalie_boundaries = pd.concat(
    [
        goalie_shifts_df[
            ["gameId", "period", "period_start_time"]
        ].rename(
            columns={"period_start_time": "boundary_time"}
        ),

        goalie_shifts_df[
            ["gameId", "period", "period_end_time"]
        ].rename(
            columns={"period_end_time": "boundary_time"}
        )
    ],
    ignore_index=True
).drop_duplicates()

goalie_boundaries["already_skater_boundary"] = [
    (game_id, period, boundary_time) in skater_boundary_keys
    for game_id, period, boundary_time in zip(
        goalie_boundaries["gameId"],
        goalie_boundaries["period"],
        goalie_boundaries["boundary_time"]
    )
]

print("Goalie boundaries:", len(goalie_boundaries))

print(
    "Additional goalie-only boundaries:",
    (~goalie_boundaries["already_skater_boundary"]).sum()
)

Goalie boundaries: 12642
Additional goalie-only boundaries: 16


### Inspect Goaltender-Only Stint Boundaries

Most goaltender shift boundaries coincide with a skater shift boundary.

A small number of goaltender boundaries occur at times when no skater substitution is recorded. These observations are inspected before constructing the final stint boundaries to determine whether they represent meaningful goaltender changes or irregularities in the NHL shift data.

In [33]:
goalie_only_boundaries = (
    goalie_boundaries.loc[
        ~goalie_boundaries["already_skater_boundary"]
    ]
    .sort_values(
        ["gameId", "period", "boundary_time"]
    )
    .reset_index(drop=True)
)

goalie_only_boundaries

,gameId,period,boundary_time,already_skater_boundary
0,2025020182,2,736,False
1,2025020498,2,824,False
2,2025020512,2,367,False
3,2025020512,2,394,False
4,2025020512,2,420,False
5,2025020616,4,112,False
6,2025020623,3,1191,False
7,2025020855,2,131,False
8,2025021012,3,64,False
9,2025021031,3,236,False


In [34]:
goalie_boundary_details = pd.concat(
    [
        goalie_shifts_df[
            [
                "gameId",
                "period",
                "playerId",
                "firstName",
                "lastName",
                "teamAbbrev",
                "period_start_time"
            ]
        ].rename(
            columns={"period_start_time": "boundary_time"}
        ).assign(boundary_type="shift_start"),

        goalie_shifts_df[
            [
                "gameId",
                "period",
                "playerId",
                "firstName",
                "lastName",
                "teamAbbrev",
                "period_end_time"
            ]
        ].rename(
            columns={"period_end_time": "boundary_time"}
        ).assign(boundary_type="shift_end")
    ],
    ignore_index=True
)

goalie_only_details = (
    goalie_only_boundaries[
        ["gameId", "period", "boundary_time"]
    ]
    .merge(
        goalie_boundary_details,
        on=["gameId", "period", "boundary_time"],
        how="left"
    )
    .sort_values(
        ["gameId", "period", "boundary_time", "teamAbbrev"]
    )
    .reset_index(drop=True)
)

goalie_only_details

,gameId,period,boundary_time,playerId,firstName,lastName,teamAbbrev,boundary_type
0,2025020182,2,736,8481668,Arturs,Silovs,PIT,shift_end
1,2025020498,2,824,8477990,Brandon,Halverson,TBL,shift_start
2,2025020498,2,824,8477992,Jonas,Johansson,TBL,shift_end
3,2025020512,2,367,8475717,Calvin,Pickard,EDM,shift_end
4,2025020512,2,394,8482487,Jakub,Dobes,MTL,shift_end
5,2025020512,2,420,8482487,Jakub,Dobes,MTL,shift_end
6,2025020616,4,112,8478916,Joey,Daccord,SEA,shift_end
7,2025020623,3,1191,8481020,Justus,Annunen,NSH,shift_end
8,2025020855,2,131,8484268,Nikita,Tolopilo,VAN,shift_start
9,2025020855,2,131,8480947,Kevin,Lankinen,VAN,shift_end


### Construct Final Stint Boundaries

A stint is defined as a continuous interval of play during which no player substitutions occur.

Initial candidate boundaries were identified from skater shift start and end times. Goaltender shift boundaries were then compared with the skater boundaries. Most goaltender boundaries occurred at times already represented by a skater substitution, but 16 additional goalie-only boundary times were identified.

Inspection showed that several of these observations corresponded to goaltender substitutions occurring without a simultaneous skater change. Therefore, both skater and goaltender shift start and end times are used to construct the final set of candidate stint boundaries.

Duplicate boundary times within the same game and period are removed so that each unique player-change time defines a single boundary.

In [35]:
stint_boundaries = (
    pd.concat(
        [
            # Skater shift starts
            skater_shifts_df[
                ["gameId", "period", "period_start_time"]
            ].rename(
                columns={"period_start_time": "boundary_time"}
            ),

            # Skater shift ends
            skater_shifts_df[
                ["gameId", "period", "period_end_time"]
            ].rename(
                columns={"period_end_time": "boundary_time"}
            ),

            # Goalie shift starts
            goalie_shifts_df[
                ["gameId", "period", "period_start_time"]
            ].rename(
                columns={"period_start_time": "boundary_time"}
            ),

            # Goalie shift ends
            goalie_shifts_df[
                ["gameId", "period", "period_end_time"]
            ].rename(
                columns={"period_end_time": "boundary_time"}
            )
        ],
        ignore_index=True
    )
    .drop_duplicates()
    .sort_values(
        ["gameId", "period", "boundary_time"]
    )
    .reset_index(drop=True)
)

print("Final candidate boundaries:", len(stint_boundaries))

Final candidate boundaries: 524568


### Construct Candidate Stints

Consecutive player-change boundaries within each game and period are used to define candidate stints.

For each boundary, the next boundary within the same game and period becomes the end of the stint. Stint duration is calculated as the difference between the start and end times.

Because each boundary represents a potential change in the on-ice personnel, the interval between consecutive boundaries should contain a constant set of players.

In [36]:
candidate_stints_df = stint_boundaries.copy()

candidate_stints_df["stint_start"] = (
    candidate_stints_df["boundary_time"]
)

candidate_stints_df["stint_end"] = (
    candidate_stints_df
    .groupby(["gameId", "period"])["boundary_time"]
    .shift(-1)
)

# Final boundary of each period has no following interval
candidate_stints_df = (
    candidate_stints_df.loc[
        candidate_stints_df["stint_end"].notna()
    ]
    .copy()
    .reset_index(drop=True)
)

candidate_stints_df["stint_duration"] = (
    candidate_stints_df["stint_end"]
    - candidate_stints_df["stint_start"]
)

candidate_stints_df.head(20)

,gameId,period,boundary_time,stint_start,stint_end,stint_duration
0,2025020001,1,0,0,34.0,34.0
1,2025020001,1,34,34,39.0,5.0
2,2025020001,1,39,39,57.0,18.0
3,2025020001,1,57,57,62.0,5.0
4,2025020001,1,62,62,65.0,3.0
5,2025020001,1,65,65,66.0,1.0
6,2025020001,1,66,66,75.0,9.0
7,2025020001,1,75,75,92.0,17.0
8,2025020001,1,92,92,106.0,14.0
9,2025020001,1,106,106,139.0,33.0


In [37]:
print("Candidate stints:", len(candidate_stints_df))

print(
    "Zero/negative-duration stints:",
    (candidate_stints_df["stint_duration"] <= 0).sum()
)

candidate_stints_df["stint_duration"].describe()

Candidate stints: 520306
Zero/negative-duration stints: 0


count    520306.000000
mean          9.204616
std          12.040830
min           1.000000
25%           1.000000
50%           4.000000
75%          12.000000
max         211.000000
Name: stint_duration, dtype: float64

### Validate On-Ice Lineup Construction

Before assigning players to every stint in the season, the lineup reconstruction procedure is tested on a single game.

For each candidate stint, a skater is considered on the ice when the player's recorded shift covers the complete stint interval. Because candidate stint boundaries were created from all player shift starts and ends, the set of active players should remain constant within each stint.

The reconstructed lineups are inspected before applying the procedure to the full dataset.

# TEST START

In [38]:
test_game_id = 2025020001

test_stints = (
    candidate_stints_df.loc[
        candidate_stints_df["gameId"] == test_game_id
    ]
    .copy()
    .reset_index(drop=True)
)

test_shifts = (
    skater_shifts_df.loc[
        skater_shifts_df["gameId"] == test_game_id
    ]
    .copy()
    .reset_index(drop=True)
)

print("Test stints:", len(test_stints))
print("Test skater shifts:", len(test_shifts))

Test stints: 403
Test skater shifts: 850


In [39]:
def get_stint_skaters(stint, shifts):
    
    active_shifts = shifts.loc[
        (shifts["period"] == stint["period"])
        & (shifts["period_start_time"] <= stint["stint_start"])
        & (shifts["period_end_time"] >= stint["stint_end"])
    ]

    lineups = (
        active_shifts
        .groupby("teamAbbrev")["playerId"]
        .apply(lambda x: tuple(sorted(x.astype(int).unique())))
        .to_dict()
    )

    return lineups

In [40]:
test_stints["lineups"] = test_stints.apply(
    lambda stint: get_stint_skaters(
        stint,
        test_shifts
    ),
    axis=1
)

test_stints[
    [
        "gameId",
        "period",
        "stint_start",
        "stint_end",
        "stint_duration",
        "lineups"
    ]
].head(30)

,gameId,period,stint_start,stint_end,stint_duration,lineups
0,2025020001,1,0,34.0,34.0,"{'CHI': (8476473, 8477450, 8481624, 8482703, 8..."
1,2025020001,1,34,39.0,5.0,"{'CHI': (8476882, 8477450, 8477479, 8482176, 8..."
2,2025020001,1,39,57.0,18.0,"{'CHI': (8476882, 8477479, 8482176, 8483493, 8..."
3,2025020001,1,57,62.0,5.0,"{'CHI': (8476882, 8476891, 8477479, 8483493, 8..."
4,2025020001,1,62,65.0,3.0,"{'CHI': (8476882, 8476891, 8477479, 8483493, 8..."
5,2025020001,1,65,66.0,1.0,"{'CHI': (8476882, 8476891, 8477479, 8483493, 8..."
6,2025020001,1,66,75.0,9.0,"{'CHI': (8476891, 8477479, 8477987, 8483493, 8..."
7,2025020001,1,75,92.0,17.0,"{'CHI': (8476891, 8477444, 8477987, 8483506, 8..."
8,2025020001,1,92,106.0,14.0,"{'CHI': (8476891, 8477444, 8477987, 8483506, 8..."
9,2025020001,1,106,139.0,33.0,"{'CHI': (8473422, 8476473, 8478043, 8482807, 8..."


In [41]:
def lineup_sizes(lineups):
    return {
        team: len(players)
        for team, players in lineups.items()
    }

test_stints["lineup_sizes"] = (
    test_stints["lineups"]
    .apply(lineup_sizes)
)

test_stints[
    [
        "period",
        "stint_start",
        "stint_end",
        "lineup_sizes"
    ]
].head(50)

,period,stint_start,stint_end,lineup_sizes
0,1,0,34.0,"{'CHI': 5, 'FLA': 5}"
1,1,34,39.0,"{'CHI': 5, 'FLA': 5}"
2,1,39,57.0,"{'CHI': 5, 'FLA': 5}"
3,1,57,62.0,"{'CHI': 5, 'FLA': 5}"
4,1,62,65.0,"{'CHI': 5, 'FLA': 5}"
5,1,65,66.0,"{'CHI': 5, 'FLA': 5}"
6,1,66,75.0,"{'CHI': 5, 'FLA': 5}"
7,1,75,92.0,"{'CHI': 5, 'FLA': 5}"
8,1,92,106.0,"{'CHI': 5, 'FLA': 5}"
9,1,106,139.0,"{'CHI': 5, 'FLA': 5}"


In [42]:
test_stints["total_skaters"] = (
    test_stints["lineups"]
    .apply(
        lambda x: sum(len(players) for players in x.values())
    )
)

test_stints["total_skaters"].value_counts().sort_index()

total_skaters
9      49
10    353
11      1
Name: count, dtype: int64

# TEST END

## Construct Skater Lineups for All Stints

In [43]:
from tqdm.auto import tqdm

all_stint_lineups = []

shift_groups = {
    key: group
    for key, group in skater_shifts_df.groupby(
        ["gameId", "period"],
        sort=False
    )
}

for (game_id, period), stints in tqdm(
    candidate_stints_df.groupby(
        ["gameId", "period"],
        sort=False
    ),
    desc="Reconstructing skater lineups"
):

    shifts = shift_groups.get((game_id, period))

    if shifts is None:
        continue

    for idx, stint in stints.iterrows():

        active = shifts.loc[
            (shifts["period_start_time"] <= stint["stint_start"])
            & (shifts["period_end_time"] >= stint["stint_end"])
        ]

        for team, team_shifts in active.groupby("teamAbbrev"):

            players = tuple(
                sorted(
                    team_shifts["playerId"]
                    .astype(int)
                    .unique()
                )
            )

            all_stint_lineups.append(
                {
                    "stint_index": idx,
                    "teamAbbrev": team,
                    "players": players,
                    "n_skaters": len(players)
                }
            )

Reconstructing skater lineups:   0%|          | 0/4262 [00:00<?, ?it/s]

In [44]:
stint_lineups_df = pd.DataFrame(all_stint_lineups)

print("Lineup records:", len(stint_lineups_df))

stint_lineups_df.head(10)

Lineup records: 1042007


,stint_index,teamAbbrev,players,n_skaters
0,0,CHI,"(8476473, 8477450, 8481624, 8482703, 8482807)",5
1,0,FLA,"(8473419, 8477409, 8477495, 8477935, 8478859)",5
2,1,CHI,"(8476882, 8477450, 8477479, 8482176, 8484783)",5
3,1,FLA,"(8473507, 8475179, 8477933, 8480185, 8482113)",5
4,2,CHI,"(8476882, 8477479, 8482176, 8483493, 8484783)",5
5,2,FLA,"(8473507, 8475179, 8477933, 8480185, 8482113)",5
6,3,CHI,"(8476882, 8476891, 8477479, 8483493, 8484783)",5
7,3,FLA,"(8473507, 8475179, 8477933, 8480185, 8482113)",5
8,4,CHI,"(8476882, 8476891, 8477479, 8483493, 8484783)",5
9,4,FLA,"(8477932, 8477933, 8478055, 8480185, 8482113)",5


In [45]:
stint_lineups_df["n_skaters"].value_counts().sort_index()

n_skaters
2          1
3       9852
4      68297
5     956438
6       7367
7         30
8          9
9          6
10         7
Name: count, dtype: int64

### Validate Reconstructed Skater Lineups

The reconstructed lineups are examined to verify that candidate stints contain the expected number of team lineups and skaters.

Most team lineups contain three, four, or five skaters, corresponding to the primary strength states used in the RAPM models. Six-skater lineups and a small number of other unusual lineup sizes are retained at this stage and will be handled when the supported RAPM strength states are defined.

Each candidate stint is also checked to confirm that lineup information was reconstructed for both teams before the lineups are converted into home and away representations.

In [47]:
lineups_per_stint = (
    stint_lineups_df
    .groupby("stint_index")
    .size()
)

lineups_per_stint.value_counts().sort_index()

1         1
2    519605
4       699
Name: count, dtype: int64

In [48]:
print("Candidate stints:", len(candidate_stints_df))
print("Stints with lineup records:", stint_lineups_df["stint_index"].nunique())

Candidate stints: 520306
Stints with lineup records: 520305


### Validate Stint Team Representation

Before converting the reconstructed lineups into home and away representations, each candidate stint is checked to determine how many team lineups were identified.

A valid stint should normally contain lineup information for exactly two teams. Stints with fewer or more than two team records are inspected separately to identify potential shift-data anomalies, team-label inconsistencies, or missing lineup information.

These observations are retained during validation rather than removed automatically. Once the source of the unusual team counts is understood, the lineup data can be cleaned and converted into home and away lineups for each stint.

In [49]:
four_team_stints = lineups_per_stint[
    lineups_per_stint == 4
].index

stint_lineups_df.loc[
    stint_lineups_df["stint_index"].isin(four_team_stints),
    [
        "stint_index",
        "teamAbbrev",
        "players",
        "n_skaters"
    ]
].head(40)

,stint_index,teamAbbrev,players,n_skaters
447440,223720,BUF,"(8479420, 8480807, 8480839, 8481522, 8482659)",5
447441,223720,NJD,"(8477488, 8478414, 8479407, 8480002, 8482684)",5
447442,223720,SJS,"(8477505, 8479316, 8480172, 8481042, 8482166)",5
447443,223720,VGK,"(8474565, 8476448, 8476881, 8478396, 8478434)",5
447444,223721,BUF,"(8479420, 8480807, 8480839, 8481522, 8482659)",5
447445,223721,NJD,"(8477015, 8477488, 8478414, 8480002, 8482684)",5
447446,223721,SJS,"(8477505, 8479316, 8480172, 8481042, 8482166)",5
447447,223721,VGK,"(8474565, 8476448, 8476881, 8478396, 8478434)",5
447448,223722,BUF,"(8480807, 8480839, 8481522, 8482659, 8483500)",5
447449,223722,NJD,"(8477015, 8477488, 8478414, 8480002, 8482684)",5


In [50]:
problem_indices = (
    lineups_per_stint[
        lineups_per_stint == 4
    ]
    .index
)

candidate_stints_df.loc[
    problem_indices,
    [
        "gameId",
        "period",
        "stint_start",
        "stint_end"
    ]
].head(20)

,gameId,period,stint_start,stint_end
stint_index,,,,
223720,2025020565,1,0,29.0
223721,2025020565,1,29,35.0
223722,2025020565,1,35,36.0
223723,2025020565,1,36,37.0
223724,2025020565,1,37,38.0
223725,2025020565,1,38,39.0
223726,2025020565,1,39,40.0
223727,2025020565,1,40,46.0
223728,2025020565,1,46,48.0


In [51]:
problem_game = (
    candidate_stints_df.loc[
        problem_indices[0],
        "gameId"
    ]
)

print("Problem game:", problem_game)

skater_shifts_df.loc[
    skater_shifts_df["gameId"] == problem_game,
    "teamAbbrev"
].value_counts()

Problem game: 2025020565


teamAbbrev
NJD    375
BUF    375
VGK    340
SJS    336
Name: count, dtype: int64

### Unique Abbreviations

Checking to see if each game a team has two unique abbreviations.

In [52]:
teams_per_game = (
    skater_shifts_df
    .groupby("gameId")["teamAbbrev"]
    .nunique()
)

teams_per_game.value_counts().sort_index()

teamAbbrev
2    1311
4       1
Name: count, dtype: int64

In [53]:
problem_games = (
    teams_per_game[
        teams_per_game != 2
    ]
)

print("Problem games:", len(problem_games))

problem_games

(
    skater_shifts_df.loc[
        skater_shifts_df["gameId"].isin(problem_games.index)
    ]
    .groupby("gameId")["teamAbbrev"]
    .unique()
)

Problem games: 1


gameId
2025020565    [VGK, NJD, SJS, BUF]
Name: teamAbbrev, dtype: object

In [55]:
shots_df.loc[
    shots_df["nhl_game_id"] == 2025020565,
    [
        "nhl_game_id",
        "homeTeamCode",
        "awayTeamCode"
    ]
].drop_duplicates()

,nhl_game_id,homeTeamCode,awayTeamCode
46567,2025020565,NJD,BUF


In [56]:
vgk_sjs_games = (
    shots_df.loc[
        (
            (shots_df["homeTeamCode"] == "VGK")
            & (shots_df["awayTeamCode"] == "SJS")
        )
        |
        (
            (shots_df["homeTeamCode"] == "SJS")
            & (shots_df["awayTeamCode"] == "VGK")
        ),
        [
            "nhl_game_id",
            "homeTeamCode",
            "awayTeamCode"
        ]
    ]
    .drop_duplicates()
    .sort_values("nhl_game_id")
    .reset_index(drop=True)
)

vgk_sjs_games

,nhl_game_id,homeTeamCode,awayTeamCode
0,2025020020,SJS,VGK
1,2025020400,VGK,SJS
2,2025020586,VGK,SJS
3,2025020715,SJS,VGK


#### Step 1: Compare player IDs

In [57]:
# VGK and SJS players incorrectly assigned to game 2025020565
problem_players = set(
    skater_shifts_df.loc[
        (skater_shifts_df["gameId"] == 2025020565)
        & (skater_shifts_df["teamAbbrev"].isin(["VGK", "SJS"])),
        "playerId"
    ].astype(int)
)

# Compare with the four possible games
player_comparison = []

for game_id in vgk_sjs_games["nhl_game_id"]:

    game_players = set(
        skater_shifts_df.loc[
            (skater_shifts_df["gameId"] == game_id)
            & (skater_shifts_df["teamAbbrev"].isin(["VGK", "SJS"])),
            "playerId"
        ].astype(int)
    )

    player_comparison.append({
        "gameId": game_id,
        "matching_players": len(problem_players & game_players),
        "problem_players": len(problem_players),
        "candidate_players": len(game_players),
        "exact_match": problem_players == game_players
    })

pd.DataFrame(player_comparison)

,gameId,matching_players,problem_players,candidate_players,exact_match
0,2025020020,19,36,36,False
1,2025020400,20,36,36,False
2,2025020586,16,36,36,False
3,2025020715,15,36,36,False


#### Duplicate Shifts

Rather than continuing to guess the game ID, let's compare the actual shift records against those from the four candidate games.

In [58]:
problem_shifts = skater_shifts_df.loc[
    (skater_shifts_df["gameId"] == 2025020565)
    & (skater_shifts_df["teamAbbrev"].isin(["VGK", "SJS"]))
].copy()

comparison = []

for game_id in vgk_sjs_games["nhl_game_id"]:

    candidate_shifts = skater_shifts_df.loc[
        (skater_shifts_df["gameId"] == game_id)
        & (skater_shifts_df["teamAbbrev"].isin(["VGK", "SJS"]))
    ].copy()

    match_columns = [
        "playerId",
        "teamAbbrev",
        "period",
        "period_start_time",
        "period_end_time"
    ]

    matching = problem_shifts[match_columns].merge(
        candidate_shifts[match_columns].drop_duplicates(),
        on=match_columns,
        how="inner"
    )

    comparison.append({
        "gameId": game_id,
        "problem_shift_records": len(problem_shifts),
        "candidate_shift_records": len(candidate_shifts),
        "matching_shift_records": len(matching),
        "match_percentage": round(
            100 * len(matching) / len(problem_shifts), 2
        )
    })

pd.DataFrame(comparison)

,gameId,problem_shift_records,candidate_shift_records,matching_shift_records,match_percentage
0,2025020020,676,765,0,0.00
1,2025020400,676,706,0,0.00
2,2025020586,676,728,0,0.00
3,2025020715,676,707,1,0.15


In [60]:
problem_shifts = shifts_df.loc[
    (shifts_df["gameId"] == 2025020565)
    & (shifts_df["teamAbbrev"].isin(["VGK", "SJS"]))
].copy()

problem_shifts[
    [
        "id",
        "gameId",
        "teamAbbrev",
        "playerId",
        "period",
        "shiftNumber",
        "startTime",
        "endTime"
    ]
].head(20)

,id,gameId,teamAbbrev,playerId,period,shiftNumber,startTime,endTime
428126,15970171,2025020565,VGK,8474565,1,1,00:00,00:36
428127,15970172,2025020565,VGK,8474565,1,2,02:55,04:40
428128,15970173,2025020565,VGK,8474565,1,3,06:42,07:29
428129,15970174,2025020565,VGK,8474565,1,4,08:52,09:44
428130,15970175,2025020565,VGK,8474565,1,5,12:05,12:54
428131,15970176,2025020565,VGK,8474565,1,6,14:19,15:15
428132,15970177,2025020565,VGK,8474565,1,7,16:47,17:34
428133,15970178,2025020565,VGK,8474565,1,8,18:38,18:46
428134,15970179,2025020565,VGK,8474565,2,9,00:59,01:38
428135,15970180,2025020565,VGK,8474565,2,10,04:21,04:52


In [61]:
(
    shifts_df.loc[
        shifts_df["gameId"] == 2025020565
    ]
    .groupby("teamAbbrev")["id"]
    .agg(["min", "max", "count"])
)

,min,max,count
teamAbbrev,,,
BUF,15964400,15971591,378
NJD,15964396,15970072,380
SJS,15970153,15970837,336
VGK,15970152,15970777,340


In [62]:
suspect_game_id = 2025020565
suspect_teams = ["VGK", "SJS"]

suspect_shifts_df = shifts_df.loc[
    (shifts_df["gameId"] == suspect_game_id)
    & (shifts_df["teamAbbrev"].isin(suspect_teams))
].copy()

print("Quarantined shift records:", len(suspect_shifts_df))

suspect_shifts_df["teamAbbrev"].value_counts()

Quarantined shift records: 676


teamAbbrev
VGK    340
SJS    336
Name: count, dtype: int64

### Remove Incorrectly Associated Shift Records

During lineup validation, game `2025020565` was found to contain shift records for four teams: NJD, BUF, VGK, and SJS.

The shot dataset identifies NJD and BUF as the participating teams. Further investigation identified 676 VGK–SJS shift records incorrectly associated with this game ID.

Comparisons against the four VGK–SJS games in the dataset did not establish a valid alternative game assignment.

The 676 records are therefore quarantined and excluded from the working shift dataset. The affected stint boundaries and lineups will be reconstructed using the corrected data.

In [63]:
shifts_df = (
    shifts_df.loc[
        ~(
            (shifts_df["gameId"] == 2025020565)
            & (shifts_df["teamAbbrev"].isin(["VGK", "SJS"]))
        )
    ]
    .copy()
    .reset_index(drop=True)
)

In [64]:
print("Remaining shift records:", len(shifts_df))

print(
    "\nTeams in game 2025020565:"
)

print(
    shifts_df.loc[
        shifts_df["gameId"] == 2025020565,
        "teamAbbrev"
    ].value_counts()
)

print(
    "\nGames with an unexpected number of teams:"
)

print(
    shifts_df.groupby("gameId")["teamAbbrev"]
    .nunique()
    .loc[lambda x: x != 2]
)

Remaining shift records: 991103

Teams in game 2025020565:
teamAbbrev
NJD    380
BUF    378
Name: count, dtype: int64

Games with an unexpected number of teams:
Series([], Name: teamAbbrev, dtype: int64)


In [65]:
skater_shifts_df = (
    shifts_df.loc[
        ~shifts_df["is_goalie"]
    ]
    .copy()
    .reset_index(drop=True)
)

goalie_shifts_df = (
    shifts_df.loc[
        shifts_df["is_goalie"]
    ]
    .copy()
    .reset_index(drop=True)
)

print("Skater shifts:", len(skater_shifts_df))
print("Goalie shifts:", len(goalie_shifts_df))

Skater shifts: 980619
Goalie shifts: 10484


In [66]:
lineups_per_stint = (
    stint_lineups_df
    .groupby("stint_index")
    .size()
)

print("Candidate stints:", len(candidate_stints_df))

print(
    "Stints with lineup records:",
    stint_lineups_df["stint_index"].nunique()
)

print("\nTeam lineups per stint:")
print(lineups_per_stint.value_counts().sort_index())

Candidate stints: 520306
Stints with lineup records: 520305

Team lineups per stint:
1         1
2    519605
4       699
Name: count, dtype: int64
